# Distribution performance benchmarks

This notebook compares Baldr with `scipy.stats` for `logpdf`, `pdf`,
`cdf`, and `ppf`. It keeps scalar sampler calls separate from array
workloads and separates JAX compilation from warm execution.

Absolute timings are specific to the machine and installed package
versions. The useful result is the relative behaviour within one run.
Set `PROFILE = "full"` for a more stable, longer benchmark.

In [ ]:
import importlib.metadata
import platform
import statistics
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

import baldr
from baldr import Beta, Gamma, Normal, TruncatedNormal

try:
    import jax
    import jax.numpy as jnp
except ImportError:
    jax = None

PROFILE = "quick"  # Change to "full" for publication-quality runs.
REPEAT = 5 if PROFILE == "quick" else 15
NUMBER_SCALAR = 2_000 if PROFILE == "quick" else 50_000
NUMBER_ARRAY = 50 if PROFILE == "quick" else 500
TARGET_ELEMENTS = 200_000 if PROFILE == "quick" else 2_000_000
ARRAY_SIZES = (8, 256, 4_096) if PROFILE == "quick" else (
    8, 32, 256, 4_096, 65_536, 1_000_000
)

In [ ]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


metadata = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.processor() or "not reported",
    **{
        name: package_version(name)
        for name in ("baldr", "numpy", "scipy", "jax", "jaxlib")
    },
}
pd.Series(metadata, name="version")

## Benchmark definitions

Construction is deliberately excluded from warm-call timings: inference
code normally constructs a prior once and evaluates it many times. JAX
results are synchronized with `block_until_ready()`.

In [ ]:
def synchronize(value):
    blocker = getattr(value, "block_until_ready", None)
    return blocker() if blocker is not None else value


def time_call(function, argument, *, repeat, number, synchronize_result=False):
    samples = []
    for _ in range(repeat):
        start = time.perf_counter_ns()
        for _ in range(number):
            result = function(argument)
            if synchronize_result:
                synchronize(result)
        samples.append((time.perf_counter_ns() - start) / number)
    return {
        "median_ns": statistics.median(samples),
        "minimum_ns": min(samples),
        "maximum_ns": max(samples),
    }


SPECS = {
    "Normal": {
        "baldr": lambda backend: Normal(
            loc=0.3, scale=1.7, backend=backend
        ),
        "scipy": lambda: stats.norm(loc=0.3, scale=1.7),
        "x": 0.8,
    },
    "Beta": {
        "baldr": lambda backend: Beta(
            a=2.3, b=5.1, loc=-0.2, scale=1.4, backend=backend
        ),
        "scipy": lambda: stats.beta(
            a=2.3, b=5.1, loc=-0.2, scale=1.4
        ),
        "x": 0.35,
    },
    "Gamma": {
        "baldr": lambda backend: Gamma(
            a=2.7, loc=0.1, scale=1.3, backend=backend
        ),
        "scipy": lambda: stats.gamma(
            a=2.7, loc=0.1, scale=1.3
        ),
        "x": 2.1,
    },
    "TruncatedNormal": {
        "baldr": lambda backend: TruncatedNormal(
            loc=0.2,
            scale=1.1,
            low=-1.4,
            high=2.7,
            backend=backend,
        ),
        "scipy": lambda: stats.truncnorm(
            (-1.4 - 0.2) / 1.1,
            (2.7 - 0.2) / 1.1,
            loc=0.2,
            scale=1.1,
        ),
        "x": 0.6,
    },
}

METHODS = ("logpdf", "pdf", "cdf", "ppf")

## Repeated scalar calls

This is the Dynesty-like case: Python repeatedly calls a distribution
with one scalar. The scalar backend is expected to be most competitive
here because array creation and framework dispatch cannot be amortized.

In [ ]:
scalar_rows = []
for distribution_name, spec in SPECS.items():
    objects = {
        "baldr_scalar": spec["baldr"]("scalar"),
        "baldr_numpy": spec["baldr"]("numpy"),
        "scipy_stats": spec["scipy"](),
    }
    for method_name in METHODS:
        argument = 0.37 if method_name == "ppf" else spec["x"]
        for implementation, distribution in objects.items():
            timing = time_call(
                getattr(distribution, method_name),
                argument,
                repeat=REPEAT,
                number=NUMBER_SCALAR,
            )
            scalar_rows.append(
                {
                    "distribution": distribution_name,
                    "method": method_name,
                    "implementation": implementation,
                    **timing,
                }
            )

scalar_results = pd.DataFrame(scalar_rows)
scalar_results.pivot_table(
    index=["distribution", "method"],
    columns="implementation",
    values="median_ns",
).round(1)

In [ ]:
normal_scalar = scalar_results.query("distribution == 'Normal'")
plot = normal_scalar.pivot(
    index="method", columns="implementation", values="median_ns"
)
plot.plot.bar(figsize=(9, 4), logy=True)
plt.ylabel("Median time per call [ns]")
plt.title("Normal scalar-call performance")
plt.xticks(rotation=0)
plt.tight_layout()

## Vector workloads

Here the input size varies while the distribution parameters remain
scalar. Times are reported both per call and per element; the crossover
in per-element cost is more informative than a single large-array result.

In [ ]:
array_rows = []
for distribution_name, spec in SPECS.items():
    baldr_numpy = spec["baldr"]("numpy")
    scipy_distribution = spec["scipy"]()
    for size in ARRAY_SIZES:
        x = np.linspace(0.05, 0.95, size)
        values = (
            x
            if distribution_name == "Beta"
            else np.linspace(spec["x"] - 0.5, spec["x"] + 0.5, size)
        )
        for method_name in METHODS:
            argument = x if method_name == "ppf" else values
            for implementation, distribution in (
                ("baldr_numpy", baldr_numpy),
                ("scipy_stats", scipy_distribution),
            ):
                timing = time_call(
                    getattr(distribution, method_name),
                    argument,
                    repeat=REPEAT,
                    number=max(
                        1,
                        min(
                            NUMBER_ARRAY,
                            TARGET_ELEMENTS // size,
                        ),
                    ),
                )
                array_rows.append(
                    {
                        "distribution": distribution_name,
                        "method": method_name,
                        "implementation": implementation,
                        "size": size,
                        **timing,
                    }
                )

array_results = pd.DataFrame(array_rows)
array_results["median_ns_per_element"] = (
    array_results["median_ns"] / array_results["size"]
)
array_results.head()

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for axis, method_name in zip(axes.flat, METHODS):
    subset = array_results.query(
        "distribution == 'Normal' and method == @method_name"
    )
    for implementation, group in subset.groupby("implementation"):
        axis.loglog(
            group["size"],
            group["median_ns_per_element"],
            marker="o",
            label=implementation,
        )
    axis.set_title(method_name)
    axis.set_ylabel("Median time [ns / element]")
    axis.set_xlabel("Array size")
axes[0, 0].legend()
figure.suptitle("Normal vector performance")
figure.tight_layout()

## JAX: eager, method-JIT, and compilation cost

JAX methods are traceable but are not internally JIT compiled by Baldr.
Compilation is measured once and kept separate from warm calls. For real
inference, compile the complete prior transform or likelihood rather than
treating each method as a separate compiled island.

In [ ]:
jax_rows = []
if jax is not None:
    for size in ARRAY_SIZES:
        x = jnp.linspace(-0.5, 1.5, size, dtype=jnp.float64)
        for method_name in METHODS:
            argument = jnp.linspace(0.05, 0.95, size) if (
                method_name == "ppf"
            ) else x
            distribution = Normal(
                loc=0.3, scale=1.7, backend="jax"
            )
            eager = getattr(distribution, method_name)
            compiled = jax.jit(eager)

            start = time.perf_counter_ns()
            synchronize(compiled(argument))
            compilation_ns = time.perf_counter_ns() - start

            for implementation, function in (
                ("baldr_jax_eager", eager),
                ("baldr_jax_jit", compiled),
            ):
                timing = time_call(
                    function,
                    argument,
                    repeat=REPEAT,
                    number=max(
                        1,
                        min(
                            NUMBER_ARRAY,
                            TARGET_ELEMENTS // size,
                        ),
                    ),
                    synchronize_result=True,
                )
                jax_rows.append(
                    {
                        "method": method_name,
                        "implementation": implementation,
                        "size": size,
                        "compilation_ns": (
                            compilation_ns
                            if implementation == "baldr_jax_jit"
                            else 0
                        ),
                        **timing,
                    }
                )
    jax_results = pd.DataFrame(jax_rows)
    display(jax_results.head())
else:
    print("JAX is not installed; skipping this section.")

In [ ]:
if jax_rows:
    figure, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    for axis, method_name in zip(axes.flat, METHODS):
        subset = jax_results.query("method == @method_name")
        for implementation, group in subset.groupby("implementation"):
            axis.loglog(
                group["size"],
                group["median_ns"] / group["size"],
                marker="o",
                label=implementation,
            )
        axis.set_title(method_name)
        axis.set_ylabel("Median time [ns / element]")
        axis.set_xlabel("Array size")
    axes[0, 0].legend()
    figure.suptitle("Normal JAX warm-call performance")
    figure.tight_layout()

## Interpreting a run

- Use the scalar table for sampler callbacks that pass one Python float.
- Use the vector curves to locate the workload size where array overhead
  is amortized.
- Compare JAX warm timings only after checking compilation separately.
- Re-run on the deployment machine and with the dtype used by the model.
- Treat small timing differences as noise unless they persist across
  repetitions and realistic end-to-end workloads.